# Data Cleaning Pipeline

**Input:**  `data/category_cache/*.parquet` (raw, produced by `download_amazon_large.py`)  
**Output:** `data/train.parquet`, `data/test.parquet`, `models/label_encoder.pkl`

Steps:
1. Load all category parquet files
2. Apply UNSPSC segment mapping
3. Build and clean combined text field
4. Deduplicate and filter
5. Label encode and stratified split
6. Save


In [ ]:
import os
import re
import pandas as pd

PROJ      = '/home/tayana-gpu/ML/project_01_product_classifier'
CACHE_DIR = f'{PROJ}/data/category_cache'

cache_files = sorted(f for f in os.listdir(CACHE_DIR) if f.endswith('.parquet'))
print(f'Category files found: {len(cache_files)}')
for f in cache_files:
    n = len(pd.read_parquet(f'{CACHE_DIR}/{f}', columns=['parent_asin']))
    print(f'  {f:<60} {n:>10,} rows')

In [ ]:
# UNSPSC Segment mapping — keyed on HuggingFace config name (filename without .parquet)
CATEGORY_TO_UNSPSC = {
    'raw_meta_All_Beauty':                  'Pharmaceuticals and Healthcare Products',
    'raw_meta_Amazon_Fashion':              'Apparel and Luggage and Personal Care Products',
    'raw_meta_Appliances':                  'Domestic Appliances and Supplies and Consumer Electronic Products',
    'raw_meta_Arts_Crafts_and_Sewing':      'Office Equipment and Accessories and Supplies',
    'raw_meta_Automotive':                  'Transportation and Storage and Mail Services',
    'raw_meta_Baby_Products':               'Pharmaceuticals and Healthcare Products',
    'raw_meta_Beauty_and_Personal_Care':    'Pharmaceuticals and Healthcare Products',
    'raw_meta_Books':                       'Printed Media',
    'raw_meta_CDs_and_Vinyl':               'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Cell_Phones_and_Accessories': 'Electronic Components and Supplies',
    'raw_meta_Clothing_Shoes_and_Jewelry':  'Apparel and Luggage and Personal Care Products',
    'raw_meta_Digital_Music':               'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Electronics':                 'Electronic Components and Supplies',
    'raw_meta_Grocery_and_Gourmet_Food':    'Food Beverage and Tobacco Products',
    'raw_meta_Handmade_Products':           'Apparel and Luggage and Personal Care Products',
    'raw_meta_Health_and_Household':        'Pharmaceuticals and Healthcare Products',
    'raw_meta_Health_and_Personal_Care':    'Pharmaceuticals and Healthcare Products',
    'raw_meta_Home_and_Kitchen':            'Domestic Appliances and Supplies and Consumer Electronic Products',
    'raw_meta_Industrial_and_Scientific':   'Industrial Machinery and Equipment',
    'raw_meta_Kindle_Store':                'Printed Media',
    'raw_meta_Magazine_Subscriptions':      'Printed Media',
    'raw_meta_Movies_and_TV':               'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Musical_Instruments':         'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Office_Products':             'Office Equipment and Accessories and Supplies',
    'raw_meta_Patio_Lawn_and_Garden':       'Farming and Fishing and Forestry and Wildlife Machinery',
    'raw_meta_Pet_Supplies':                'Animals and Birds and Fish',
    'raw_meta_Software':                    'Information Technology Broadcasting and Telecommunications',
    'raw_meta_Sports_and_Outdoors':         'Sports and Recreational Equipment and Supplies',
    'raw_meta_Tools_and_Home_Improvement':  'Building and Construction Machinery and Accessories',
    'raw_meta_Toys_and_Games':              'Sports and Recreational Equipment and Supplies',
    'raw_meta_Video_Games':                 'Audio and Visual Presentation and Composing Equipment',
}

print(f'Segments in mapping: {len(set(CATEGORY_TO_UNSPSC.values()))}')
print(sorted(set(CATEGORY_TO_UNSPSC.values())))

In [ ]:
# Load all cached category files and assign UNSPSC labels
frames = []
for fname in cache_files:
    config_name = fname.replace('.parquet', '')
    if config_name not in CATEGORY_TO_UNSPSC:
        print(f'WARNING: no mapping for {config_name}, skipping')
        continue
    df_cat = pd.read_parquet(f'{CACHE_DIR}/{fname}')
    df_cat['unspsc_segment'] = CATEGORY_TO_UNSPSC[config_name]
    frames.append(df_cat[['parent_asin', 'title', 'description', 'features', 'unspsc_segment']])

raw = pd.concat(frames, ignore_index=True)
print(f'Total raw rows loaded: {len(raw):,}')
print(f'\nRaw segment distribution:')
print(raw['unspsc_segment'].value_counts())

In [ ]:
# Build and clean the combined text field

def clean_text(text: str) -> str:
    """Supply-chain-aware cleaner — keeps numbers, specs, and hyphens."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)               # strip HTML tags
    text = re.sub(r'http\S+', ' ', text)                # strip URLs
    text = re.sub(r'[^a-zA-Z0-9\s\-\/\.]', ' ', text)  # keep alphanumeric + spec chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def join_list_field(field) -> str:
    if isinstance(field, list):
        return ' '.join(str(x) for x in field if x)
    if isinstance(field, str):
        return field
    return ''

raw['text'] = (
    raw['title'].apply(clean_text) + ' ' +
    raw['description'].apply(join_list_field).apply(clean_text) + ' ' +
    raw['features'].apply(join_list_field).apply(clean_text)
).str.strip()

print('Text column built.')
print(f'\nSample rows:')
for _, row in raw[['unspsc_segment', 'text']].sample(3, random_state=1).iterrows():
    print(f'  [{row["unspsc_segment"][:40]}]  {row["text"][:120]}')

In [ ]:
# Quality filters

n0 = len(raw)

# Drop rows where text is too short to carry signal
raw = raw[raw['text'].str.split().str.len() >= 3].copy()
print(f'Dropped (< 3 words):      {n0 - len(raw):>10,}')

# Drop exact text duplicates
n1 = len(raw)
raw = raw.drop_duplicates(subset=['text']).copy()
print(f'Dropped (exact duplicates):{n1 - len(raw):>10,}')

# Drop segments that do not have enough data to train reliably
counts = raw['unspsc_segment'].value_counts()
valid_segments  = counts[counts >= 500].index
dropped_segments = counts[counts < 500].index.tolist()
raw = raw[raw['unspsc_segment'].isin(valid_segments)].copy()
if dropped_segments:
    print(f'Dropped segments (< 500 rows): {dropped_segments}')

print(f'\nFinal rows:     {len(raw):,}')
print(f'Final segments: {raw["unspsc_segment"].nunique()}')

print(f'\nFinal class distribution:')
counts = raw['unspsc_segment'].value_counts()
for seg, n in counts.items():
    print(f'  {seg:<65} {n:>10,}')
print(f'\nImbalance ratio: {counts.max() / counts.min():.0f}x')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

le = LabelEncoder()
raw['label'] = le.fit_transform(raw['unspsc_segment'])

print('Label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i:>2}: {cls}')

train_df, test_df = train_test_split(
    raw[['text', 'unspsc_segment', 'label']],
    test_size=0.20,
    stratify=raw['label'],
    random_state=42
)

print(f'\nTrain: {len(train_df):,} | Test: {len(test_df):,}')

train_df.to_parquet(f'{PROJ}/data/train.parquet', index=False)
test_df.to_parquet(f'{PROJ}/data/test.parquet', index=False)
joblib.dump(le, f'{PROJ}/models/label_encoder.pkl')

print('\nSaved:')
print('  data/train.parquet')
print('  data/test.parquet')
print('  models/label_encoder.pkl')
print('\nNext: run eda.ipynb')